# Chapter 13: Loading and Preprocessing Data with TensorFlow - Part 1

## Overview

Welcome to this practical tutorial on TensorFlow's Data API! 

**Why do we need the Data API?**

In Deep Learning, we often work with datasets that are **too large to fit in RAM**. For example:
- ImageNet contains millions of images (hundreds of GB)
- Large text corpora can be terabytes in size
- Video datasets are even larger

TensorFlow's Data API solves this problem by providing dataset objects that:
- Stream data efficiently from disk
- Handle multithreading automatically
- Manage queuing, batching, and prefetching
- Work seamlessly with tf.keras

**Data Sources supported:**
- Text files (CSV, plain text)
- Binary files with fixed-size records
- TFRecord format (TensorFlow's optimized format)
- SQL databases
- Custom sources via extensions

In this notebook, we'll cover the fundamentals of the Data API, including creating datasets, transformations, shuffling, and efficient file reading.

## 1. Import Required Libraries

First, let's import TensorFlow and other necessary libraries.

In [1]:
import tensorflow as tf
import numpy as np
import os

print(f"TensorFlow version: {tf.__version__}")

c:\Users\ASUS\anaconda3\envs\conda_env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


TensorFlow version: 2.15.0


## 2. Creating Basic Datasets

The Data API centers around the **dataset** concept - a sequence of data items.

Let's create our first dataset from a simple tensor using `tf.data.Dataset.from_tensor_slices()`.

In [2]:
# Create a simple tensor with values from 0 to 9
X = tf.range(10)
print("Original tensor:", X)
print()

# Create a dataset from the tensor
dataset = tf.data.Dataset.from_tensor_slices(X)
print("Dataset object:", dataset)
print()

# Examine the dataset properties
print("Element spec:", dataset.element_spec)

Original tensor: tf.Tensor([0 1 2 3 4 5 6 7 8 9], shape=(10,), dtype=int32)

Dataset object: <_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.int32, name=None)>

Element spec: TensorSpec(shape=(), dtype=tf.int32, name=None)


## 3. Dataset Iteration

Datasets are iterable - you can loop through them just like Python lists. Each iteration yields a single item as a tensor.

In [3]:
# Iterate over the dataset
print("Iterating over dataset items:")
for item in dataset:
    print(item)

Iterating over dataset items:
tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


## 4. Chaining Transformations - Repeat and Batch

**Key concept:** Dataset methods return NEW datasets, allowing you to chain transformations.

- **`repeat(n)`**: Repeats the dataset n times (creates n copies of the data)
- **`batch(size)`**: Groups consecutive items into batches of the specified size

⚠️ **Important:** Always save the result: `dataset = dataset.repeat(3)` (not just `dataset.repeat(3)`)

In [4]:
# Chain repeat and batch transformations
dataset = tf.data.Dataset.range(10)  # Create fresh dataset: 0-9
dataset = dataset.repeat(3)           # Repeat 3 times: 0-9, 0-9, 0-9
dataset = dataset.batch(7)            # Batch into groups of 7

print("After repeat(3) and batch(7):")
for i, batch in enumerate(dataset):
    print(f"Batch {i+1}: {batch.numpy()}")

After repeat(3) and batch(7):
Batch 1: [0 1 2 3 4 5 6]
Batch 2: [7 8 9 0 1 2 3]
Batch 3: [4 5 6 7 8 9 0]
Batch 4: [1 2 3 4 5 6 7]
Batch 5: [8 9]


**Explanation of the output:**
- We have 10 numbers repeated 3 times = 30 total items
- Batching into groups of 7 gives us 4 full batches (7 items each) and 1 partial batch (2 items)
- Notice how the repetitions wrap around within batches

## 5. Transforming Items with Map

The **`map()`** method applies a function to each item in the dataset. This is useful for:
- Data preprocessing (scaling, normalization)
- Feature engineering
- Data augmentation

**Performance tip:** Set `num_parallel_calls` to enable multithreading for CPU-intensive operations.

In [5]:
# Create a fresh dataset
dataset = tf.data.Dataset.range(10)

# Apply a transformation: multiply each element by 2
dataset = dataset.map(lambda x: x * 2)

print("After map(lambda x: x * 2):")
for item in dataset:
    print(item.numpy(), end=" ")
print("\n")

# Example with more complex transformation
dataset = tf.data.Dataset.range(5)
dataset = dataset.map(lambda x: x ** 2)  # Square each number

print("After map(lambda x: x ** 2):")
for item in dataset:
    print(f"{item.numpy()}", end=" ")
print("\n")

# Using num_parallel_calls for performance
dataset = tf.data.Dataset.range(1000)
dataset = dataset.map(lambda x: x * 2, num_parallel_calls=tf.data.AUTOTUNE)
print(f"\nWith AUTOTUNE, TensorFlow automatically optimizes parallelism.")
print(f"First 10 items: {list(dataset.take(10).as_numpy_iterator())}")

After map(lambda x: x * 2):
0 2 4 6 8 10 12 14 16 18 

After map(lambda x: x ** 2):
0 1 4 9 16 


With AUTOTUNE, TensorFlow automatically optimizes parallelism.
First 10 items: [0, 2, 4, 6, 8, 10, 12, 14, 16, 18]


## 6. Applying Transformations to Entire Datasets

While **`map()`** transforms individual items, **`apply()`** transforms the entire dataset at once.

A common use case is **`unbatch()`** - the reverse of batch().

In [6]:
# Create a batched dataset
dataset = tf.data.Dataset.range(10)
dataset = dataset.batch(3)

print("Batched dataset:")
for batch in dataset:
    print(batch.numpy())

print("\nAfter unbatch():")
# Apply unbatch to reverse the batching
dataset = dataset.unbatch()
for item in dataset:
    print(item.numpy(), end=" ")
print()

Batched dataset:
[0 1 2]
[3 4 5]
[6 7 8]
[9]

After unbatch():
0 1 2 3 4 5 6 7 8 9 


## 7. Filtering and Limiting Datasets

**`filter()`** - Keep only items that satisfy a condition  
**`take(n)`** - Take only the first n items (useful for previewing or limiting data)

In [7]:
# Create a dataset and repeat it
dataset = tf.data.Dataset.range(10).repeat(3)

# Filter: keep only values less than 10
dataset = dataset.filter(lambda x: x < 10)

print("After filter(x < 10) on repeated dataset:")
print("First 15 items using take(15):")
for item in dataset.take(15):
    print(item.numpy(), end=" ")
print("\n")

# Another example: filter even numbers
dataset = tf.data.Dataset.range(20)
even_dataset = dataset.filter(lambda x: x % 2 == 0)

print("Even numbers from 0-19:")
for item in even_dataset:
    print(item.numpy(), end=" ")
print("\n")

# Combine filter and take
print("\nFirst 5 even numbers:")
for item in even_dataset.take(5):
    print(item.numpy(), end=" ")

After filter(x < 10) on repeated dataset:
First 15 items using take(15):
0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 

Even numbers from 0-19:
0 2 4 6 8 10 12 14 16 18 


First 5 even numbers:
0 2 4 6 8 

## 8. Shuffling Data for Training

**Why shuffle?** Gradient Descent requires **independent, identically distributed (i.i.d.)** instances. If training data is ordered, the model may learn patterns related to the order rather than the features.

**How `shuffle()` works:**
- Fills a buffer with the first `buffer_size` items
- Randomly samples from this buffer
- Refills buffer as items are consumed

**Buffer size considerations:**
- Too small → poor shuffling
- Too large → exceeds RAM
- For perfect shuffling: `buffer_size` = dataset size (only for small datasets!)

In [8]:
# Demonstrate shuffling
dataset = tf.data.Dataset.range(10).repeat(3)

# Shuffle with buffer_size=5
dataset = dataset.shuffle(buffer_size=5, seed=42).batch(7)

print("After shuffle(buffer_size=5, seed=42) and batch(7):")
for i, batch in enumerate(dataset):
    print(f"Batch {i+1}: {batch.numpy()}")

print("\n" + "="*60)
print("Comparing: No shuffle vs. With shuffle")
print("="*60)

# Without shuffling
dataset_no_shuffle = tf.data.Dataset.range(10).repeat(2).batch(5)
print("\nWithout shuffle:")
for i, batch in enumerate(dataset_no_shuffle):
    print(f"Batch {i+1}: {batch.numpy()}")

# With shuffling
dataset_with_shuffle = tf.data.Dataset.range(10).repeat(2).shuffle(buffer_size=10, seed=42).batch(5)
print("\nWith shuffle (buffer_size=10):")
for i, batch in enumerate(dataset_with_shuffle):
    print(f"Batch {i+1}: {batch.numpy()}")

After shuffle(buffer_size=5, seed=42) and batch(7):
Batch 1: [0 2 3 6 7 9 4]
Batch 2: [5 0 1 1 8 6 5]
Batch 3: [4 8 7 1 2 3 0]
Batch 4: [5 4 2 7 8 9 9]
Batch 5: [3 6]

Comparing: No shuffle vs. With shuffle

Without shuffle:
Batch 1: [0 1 2 3 4]
Batch 2: [5 6 7 8 9]
Batch 3: [0 1 2 3 4]
Batch 4: [5 6 7 8 9]

With shuffle (buffer_size=10):
Batch 1: [5 2 8 1 7]
Batch 2: [9 2 0 0 4]
Batch 3: [6 9 3 5 3]
Batch 4: [8 4 1 7 6]


**Shuffling strategy for large datasets:**
1. **Shuffle source data** - randomize files on disk
2. **Split into multiple files** - read files in random order
3. **Interleave files** - read from multiple files simultaneously
4. **Add shuffle buffer** - final shuffling with reasonable buffer size

## 9. Working with CSV Files

Let's create sample CSV files to simulate a real-world scenario. We'll create housing data similar to the California Housing dataset.

In [9]:
# Create a directory for our CSV files
csv_dir = "housing_data"
if not os.path.exists(csv_dir):
    os.makedirs(csv_dir)

# Generate sample housing data
np.random.seed(42)

def generate_housing_data(n_samples):
    """Generate synthetic housing data"""
    data = {
        'MedInc': np.random.uniform(0.5, 15, n_samples).round(4),
        'HouseAge': np.random.uniform(1, 52, n_samples).round(1),
        'AveRooms': np.random.uniform(1, 10, n_samples).round(4),
        'AveBedrms': np.random.uniform(0.5, 5, n_samples).round(4),
        'Population': np.random.uniform(3, 35682, n_samples).round(1),
        'AveOccup': np.random.uniform(0.5, 6, n_samples).round(4),
        'Latitude': np.random.uniform(32.5, 42, n_samples).round(2),
        'Longitude': np.random.uniform(-124.3, -114.3, n_samples).round(2),
        'MedianHouseValue': np.random.uniform(0.5, 5, n_samples).round(3)
    }
    return data

# Create multiple CSV files
n_files = 5
samples_per_file = 100

for i in range(n_files):
    data = generate_housing_data(samples_per_file)
    filepath = os.path.join(csv_dir, f"housing_data_{i+1}.csv")
    
    # Write to CSV
    with open(filepath, 'w') as f:
        # Write header
        f.write(','.join(data.keys()) + '\\n')
        # Write data rows
        for row_idx in range(samples_per_file):
            row = [str(data[col][row_idx]) for col in data.keys()]
            f.write(','.join(row) + '\\n')
    
    print(f"Created: {filepath}")

print(f"\nTotal files created: {n_files}")
print(f"Samples per file: {samples_per_file}")
print(f"Total samples: {n_files * samples_per_file}")

Created: housing_data\housing_data_1.csv
Created: housing_data\housing_data_2.csv
Created: housing_data\housing_data_3.csv
Created: housing_data\housing_data_4.csv
Created: housing_data\housing_data_5.csv

Total files created: 5
Samples per file: 100
Total samples: 500


In [10]:
# Preview one of the CSV files
sample_file = os.path.join(csv_dir, "housing_data_1.csv")
print(f"Preview of {sample_file}:\\n")
with open(sample_file, 'r') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 4:  # Show first 5 lines (header + 4 data rows)
            break

Preview of housing_data\housing_data_1.csv:\n
MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedianHouseValue\n5.9308,2.6,6.7783,0.7326,3682.4,4.3399,34.1,-118.97,3.683\n14.2854,33.5,1.7573,2.8911,32205.2,3.4485,35.15,-123.78,1.186\n11.1139,17.0,2.4547,2.9329,18029.9,2.2024,34.18,-120.93,3.093\n9.1805,26.9,9.087,3.3684,29490.2,4.9759,33.34,-122.96,3.23\n2.7623,47.3,6.4579,3.7674,11422.0,4.266,33.65,-123.67,2.409\n2.7619,13.7,1.0828,4.8913,31954.4,1.3944,36.88,-114.4,3.814\n1.3422,21.9,1.9132,2.8234,13889.3,5.5101,34.46,-121.08,4.705\n13.0596,39.5,6.9715,1.9533,389.7,5.024,35.96,-116.2,4.665\n9.2162,12.7,1.0456,4.0783,32306.1,5.7239,37.28,-121.75,2.529\n10.7671,4.9,2.4473,1.7187,3260.0,4.4915,39.06,-117.48,1.01\n0.7985,15.8,5.9386,2.4754,11395.8,3.8738,32.87,-116.7,4.932\n14.5637,9.2,7.2271,0.8531,33900.3,2.8003,40.09,-118.34,4.275\n12.5704,48.4,6.8677,0.6141,33919.7,5.63,38.47,-119.58,1.061\n3.5789,42.2,3.0184,4.8319,20462.7,5.2634,33.28,-120.18,4.644\n3.1365

In [11]:
# Create a filepath pattern and list files
filepath_pattern = os.path.join(csv_dir, "housing_data_*.csv")
print(f"Filepath pattern: {filepath_pattern}\\n")

# Use tf.data.Dataset.list_files to create a dataset of file paths
filepath_dataset = tf.data.Dataset.list_files(filepath_pattern, seed=42)

print("File paths in dataset:")
for filepath in filepath_dataset:
    print(filepath.numpy().decode('utf-8'))

Filepath pattern: housing_data\housing_data_*.csv\n
File paths in dataset:
housing_data\housing_data_1.csv
housing_data\housing_data_5.csv
housing_data\housing_data_2.csv
housing_data\housing_data_4.csv
housing_data\housing_data_3.csv


## 10. Interleaving Multiple Files

**Problem:** Reading one file at a time is slow and doesn't utilize multiple CPU cores.

**Solution:** **Interleaving** - read from multiple files simultaneously!

**`interleave()`** parameters:
- **`cycle_length`**: Number of files to read simultaneously
- **`num_parallel_calls`**: Number of threads for parallel reading
  - Use `tf.data.AUTOTUNE` to let TensorFlow optimize automatically

**How it works:**
1. Takes `cycle_length` filepaths
2. Creates a dataset from each filepath (e.g., TextLineDataset)
3. Cycles through these datasets, taking items in a round-robin fashion

In [12]:
# Simple example: Read a single file
print("Reading a single file (first 5 lines):")
single_file_dataset = tf.data.TextLineDataset([os.path.join(csv_dir, "housing_data_1.csv")])
single_file_dataset = single_file_dataset.skip(1)  # Skip header

for i, line in enumerate(single_file_dataset.take(5)):
    print(f"Line {i+1}: {line.numpy().decode('utf-8')[:80]}...")  # First 80 chars

Reading a single file (first 5 lines):


In [13]:
# Now let's interleave multiple files
n_readers = 3  # Read from 3 files simultaneously

filepath_dataset = tf.data.Dataset.list_files(filepath_pattern, seed=42)

# Interleave: read from multiple files at once
interleaved_dataset = filepath_dataset.interleave(
    lambda filepath: tf.data.TextLineDataset(filepath).skip(1),  # skip header
    cycle_length=n_readers,
    num_parallel_calls=n_readers
)

print(f"\nInterleaved dataset (cycle_length={n_readers}):")
print("Notice how lines come from different files:")
for i, line in enumerate(interleaved_dataset.take(15)):
    line_str = line.numpy().decode('utf-8')
    # Extract first few values to identify which file it might be from
    values = line_str.split(',')[:3]
    print(f"Line {i+1:2d}: {','.join(values)}...")
    
print("\n✓ Lines are interleaved from multiple files!")


Interleaved dataset (cycle_length=3):
Notice how lines come from different files:

✓ Lines are interleaved from multiple files!


In [14]:
# Using AUTOTUNE for automatic optimization
filepath_dataset = tf.data.Dataset.list_files(filepath_pattern, seed=42)

interleaved_dataset_auto = filepath_dataset.interleave(
    lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
    cycle_length=n_readers,
    num_parallel_calls=tf.data.AUTOTUNE  # Let TensorFlow decide optimal parallelism
)

print("\\nWith AUTOTUNE:")
print("TensorFlow automatically tunes the number of parallel calls for optimal performance.")
print("\\nFirst 10 lines:")
for i, line in enumerate(interleaved_dataset_auto.take(10)):
    line_str = line.numpy().decode('utf-8')
    values = line_str.split(',')[:3]
    print(f"Line {i+1:2d}: {','.join(values)}...")

\nWith AUTOTUNE:
TensorFlow automatically tunes the number of parallel calls for optimal performance.
\nFirst 10 lines:


## Summary of Part 1

In this notebook, we covered the fundamentals of TensorFlow's Data API:

### Key Concepts Learned:

1. **Creating Datasets** - Using `from_tensor_slices()` to create datasets from tensors
2. **Chaining Transformations** - Methods like `repeat()` and `batch()` return new datasets
3. **Map Transformations** - Apply functions to individual items with `map()`
4. **Dataset Operations** - `filter()`, `take()`, `unbatch()`
5. **Shuffling** - Essential for training with proper buffer size management
6. **File Operations** - Reading CSV files with `TextLineDataset`
7. **Interleaving** - Efficiently read from multiple files in parallel

### Best Practices:

✅ Always assign transformed datasets: `dataset = dataset.batch(32)`  
✅ Use `AUTOTUNE` for automatic performance optimization  
✅ Shuffle with appropriate buffer size for training  
✅ Interleave files for parallel reading  
✅ Use `seed` parameter for reproducibility  

### What's Next (Part 2):

In the next section, we'll cover:
- Preprocessing data with `decode_csv()`
- Creating efficient data loading pipelines
- Prefetching for performance
- Using datasets with tf.keras models
- Complete end-to-end examples

---

**🎯 You now understand the core Data API concepts! Ready to build efficient data pipelines.**

---

# Part 2: Preprocessing, Optimization, and Integration with tf.keras

Now let's move to more advanced topics: preprocessing CSV data, building efficient pipelines, and training models!

## 11. Preprocessing CSV Data

When reading CSV files, we need to:
1. **Parse** each line into fields (columns)
2. **Separate** features (X) from labels (y)
3. **Standardize** features (subtract mean, divide by standard deviation)

**Key TensorFlow functions:**
- `tf.io.decode_csv()` - Parses CSV lines into tensors
- `tf.stack()` - Converts list of scalar tensors into a 1D array
- Record defaults define column types and handle missing values

In [18]:
# First, let's compute statistics from our data for standardization
# In a real scenario, you'd compute these from training data only

# Read one file to understand the data
sample_file = os.path.join(csv_dir, "housing_data_1.csv")
sample_data = []

with open(sample_file, 'r') as f:
    header = f.readline().strip().split(',')
    for line in f:
        values = [float(x) for x in line.strip().split(',')]
        sample_data.append(values)

sample_data = np.array(sample_data)
print(f"Sample data shape: {sample_data.shape}")
print(f"Features (columns): {header}")
print()

# Compute mean and std for features (all columns except last, which is the target)
n_inputs = 8  # Number of input features
X_sample = sample_data[:, :-1]  # All columns except last
y_sample = sample_data[:, -1:]   # Last column (target)

X_mean = X_sample.mean(axis=0).astype(np.float32)
X_std = X_sample.std(axis=0).astype(np.float32)

print(f"Feature means: {X_mean}")
print(f"Feature stds:  {X_std}")

Sample data shape: (0,)
Features (columns): ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedianHouseValue\\n5.9308', '2.6', '6.7783', '0.7326', '3682.4', '4.3399', '34.1', '-118.97', '3.683\\n14.2854', '33.5', '1.7573', '2.8911', '32205.2', '3.4485', '35.15', '-123.78', '1.186\\n11.1139', '17.0', '2.4547', '2.9329', '18029.9', '2.2024', '34.18', '-120.93', '3.093\\n9.1805', '26.9', '9.087', '3.3684', '29490.2', '4.9759', '33.34', '-122.96', '3.23\\n2.7623', '47.3', '6.4579', '3.7674', '11422.0', '4.266', '33.65', '-123.67', '2.409\\n2.7619', '13.7', '1.0828', '4.8913', '31954.4', '1.3944', '36.88', '-114.4', '3.814\\n1.3422', '21.9', '1.9132', '2.8234', '13889.3', '5.5101', '34.46', '-121.08', '4.705\\n13.0596', '39.5', '6.9715', '1.9533', '389.7', '5.024', '35.96', '-116.2', '4.665\\n9.2162', '12.7', '1.0456', '4.0783', '32306.1', '5.7239', '37.28', '-121.75', '2.529\\n10.7671', '4.9', '2.4473', '1.7187', '3260.0', '4.4915', '39.0

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [17]:
# Now create the preprocessing function
def preprocess(line):
    """
    Preprocess a single CSV line:
    1. Parse CSV into fields
    2. Separate features from labels
    3. Standardize features
    """
    # Define default values for each column (defines types and handles missing data)
    # 8 input features (float) + 1 target (float)
    defs = [0.] * n_inputs + [tf.constant([], dtype=tf.float32)]
    
    # Parse the CSV line
    fields = tf.io.decode_csv(line, record_defaults=defs)
    
    # Split into features (X) and target (y)
    x = tf.stack(fields[:-1])  # Stack all but last field into array
    y = tf.stack(fields[-1:])   # Stack last field
    
    # Standardize: (x - mean) / std
    x_standardized = (x - X_mean) / X_std
    
    return x_standardized, y

# Test the preprocessing function on a single line
test_line = "3.5214,15.0,3.0499,1.1065,1447.0,1.6059,37.63,-122.43,1.442"
x_processed, y_processed = preprocess(test_line)

print("Original line:")
print(test_line)
print()
print("After preprocessing:")
print(f"X (standardized features): {x_processed.numpy()}")
print(f"y (target):                {y_processed.numpy()}")

NameError: name 'X_mean' is not defined

## 12. Putting Everything Together - Efficient Data Pipeline

Now we'll create a **complete, production-ready data loading function** that combines all the techniques we've learned:

**Pipeline stages:**
1. 📁 **List files** - Create dataset of file paths
2. 🔀 **Interleave** - Read multiple files in parallel
3. 🔧 **Map (preprocess)** - Parse and standardize each line
4. 🎲 **Shuffle** - Randomize for training
5. 🔁 **Repeat** - Loop through data for multiple epochs
6. 📦 **Batch** - Group into mini-batches
7. ⚡ **Prefetch** - Prepare next batch while training current one

In [ ]:
def csv_reader_dataset(filepaths, repeat=1, n_readers=5,
                       n_read_threads=None, shuffle_buffer_size=10000,
                       n_parse_threads=5, batch_size=32):
    """
    Create an efficient dataset from CSV files.
    
    Parameters:
    -----------
    filepaths : str or list
        Pattern or list of file paths
    repeat : int
        Number of times to repeat the dataset (1 = one epoch)
    n_readers : int
        Number of files to read simultaneously
    n_read_threads : int or None
        Threads for file reading (None = use n_readers, or use AUTOTUNE)
    shuffle_buffer_size : int
        Size of shuffle buffer
    n_parse_threads : int
        Threads for parsing/preprocessing
    batch_size : int
        Number of samples per batch
    
    Returns:
    --------
    dataset : tf.data.Dataset
        Batched and prefetched dataset ready for training
    """
    # 1. List files
    dataset = tf.data.Dataset.list_files(filepaths)
    
    # 2. Interleave: read from multiple files in parallel
    dataset = dataset.interleave(
        lambda filepath: tf.data.TextLineDataset(filepath).skip(1),  # Skip header
        cycle_length=n_readers,
        num_parallel_calls=n_read_threads or tf.data.AUTOTUNE
    )
    
    # 3. Map: parse and preprocess each line
    dataset = dataset.map(preprocess, num_parallel_calls=n_parse_threads)
    
    # 4. Shuffle the data
    dataset = dataset.shuffle(shuffle_buffer_size)
    
    # 5. Repeat for multiple epochs
    dataset = dataset.repeat(repeat)
    
    # 6. Batch the data
    dataset = dataset.batch(batch_size)
    
    # 7. Prefetch: prepare next batch while training current batch
    return dataset.prefetch(1)

# Create a dataset using our function
filepath_pattern = os.path.join(csv_dir, "housing_data_*.csv")
dataset = csv_reader_dataset(filepath_pattern, repeat=1, batch_size=32)

print("✅ Created efficient dataset pipeline!")
print(f"Dataset: {dataset}")
print()

# Preview a batch
print("Sample batch:")
for X_batch, y_batch in dataset.take(1):
    print(f"X_batch shape: {X_batch.shape}")
    print(f"y_batch shape: {y_batch.shape}")
    print()
    print(f"First 3 samples (features):")
    print(X_batch[:3].numpy())
    print()
    print(f"First 3 samples (targets):")
    print(y_batch[:3].numpy())

## 13. Prefetching - Maximizing Performance

**The Problem:** While the GPU trains on one batch, the CPU sits idle. Then the GPU waits while the CPU prepares the next batch.

**The Solution:** **Prefetching** - Overlap data loading with training!

```
Without prefetch:           With prefetch(1):
CPU: [Load][Load][Load]     CPU: [Load][Load][Load]
GPU:       [Train][Train]   GPU: [Train][Train][Train]
     ^^^^^ Wasted time!          ✓ No wasted time!
```

**How it works:**
- `prefetch(1)` means "always have 1 batch ready"
- While GPU trains on batch N, CPU prepares batch N+1
- Maximizes utilization of both CPU and GPU

**Performance boost:** Combined with `num_parallel_calls=AUTOTUNE` for map and interleave, you can achieve near 100% GPU utilization!

In [ ]:
# Demonstrate the difference
print("Dataset WITHOUT prefetch:")
dataset_no_prefetch = tf.data.Dataset.range(1000).batch(32)
print(dataset_no_prefetch)
print()

print("Dataset WITH prefetch(1):")
dataset_with_prefetch = tf.data.Dataset.range(1000).batch(32).prefetch(1)
print(dataset_with_prefetch)
print()

print("Dataset WITH prefetch(AUTOTUNE):")
dataset_with_prefetch_auto = tf.data.Dataset.range(1000).batch(32).prefetch(tf.data.AUTOTUNE)
print(dataset_with_prefetch_auto)
print()

print("💡 Best Practice: Always use .prefetch(1) or .prefetch(tf.data.AUTOTUNE) at the end of your pipeline!")
print()
print("🎯 Bonus Tip: For small datasets that fit in memory, add .cache() AFTER preprocessing")
print("   but BEFORE shuffling to avoid re-preprocessing each epoch:")
print("   dataset.map(preprocess).cache().shuffle(buffer).batch(32).prefetch(1)")

## 14. Using Datasets with tf.keras

Now let's see how to use our datasets with Keras models. The Data API integrates seamlessly with tf.keras!

We'll:
1. Split data into train/validation/test sets
2. Create datasets for each split
3. Build a simple neural network
4. Train using `model.fit()` with our datasets

In [ ]:
# For this demo, we'll simulate train/valid/test splits using our existing files
# In practice, you'd have separate files for each split

# Get all file paths
all_files = sorted([os.path.join(csv_dir, f"housing_data_{i}.csv") for i in range(1, 6)])
print(f"All files: {all_files}")

# Split into train/valid/test
train_files = all_files[:3]   # First 3 files for training
valid_files = all_files[3:4]  # 4th file for validation
test_files = all_files[4:5]   # 5th file for testing

print(f"\nTrain files: {train_files}")
print(f"Valid files: {valid_files}")
print(f"Test files:  {test_files}")

In [ ]:
# Create datasets for each split
train_set = csv_reader_dataset(train_files, repeat=1, batch_size=32)
valid_set = csv_reader_dataset(valid_files, repeat=1, batch_size=32)
test_set = csv_reader_dataset(test_files, repeat=1, batch_size=32)

print("✅ Created train, validation, and test datasets!")
print(f"\nTrain set: {train_set}")
print(f"Valid set: {valid_set}")
print(f"Test set:  {test_set}")

In [ ]:
# Build a simple neural network
from tensorflow import keras

model = keras.models.Sequential([
    keras.layers.Dense(30, activation='relu', input_shape=[8]),  # 8 input features
    keras.layers.Dense(30, activation='relu'),
    keras.layers.Dense(1)  # 1 output (regression)
])

# Compile the model
model.compile(
    loss='mse',
    optimizer=keras.optimizers.SGD(learning_rate=0.01),
    metrics=['mae']
)

print("Model architecture:")
model.summary()

In [ ]:
# Train the model using our dataset!
# Notice: we pass the dataset directly to fit() - no need to extract batches manually

print("Training model...")
history = model.fit(
    train_set,
    epochs=10,
    validation_data=valid_set
)

print("\n✅ Training complete!")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_mae = model.evaluate(test_set)
print(f"\nTest Loss (MSE): {test_loss:.4f}")
print(f"Test MAE:        {test_mae:.4f}")

In [ ]:
# Make predictions on new data
# Take 3 batches from test set, but only keep features (drop labels)
new_set = test_set.take(3).map(lambda X, y: X)

print("Making predictions on 3 batches...")
predictions = model.predict(new_set)

print(f"\nPredictions shape: {predictions.shape}")
print(f"First 10 predictions:\n{predictions[:10].flatten()}")

## 15. Custom Training Loop (Advanced)

For more control, you can create a custom training loop. This is useful when you need:
- Custom gradient computations
- Multiple optimization steps per batch
- Complex loss functions
- Fine-grained control over the training process

In [ ]:
# Simple custom training loop example
print("Custom Training Loop Demo")
print("-" * 50)

# Create a fresh model
custom_model = keras.models.Sequential([
    keras.layers.Dense(30, activation='relu', input_shape=[8]),
    keras.layers.Dense(1)
])

# Setup optimizer and loss
optimizer = keras.optimizers.SGD(learning_rate=0.01)
loss_fn = keras.losses.MeanSquaredError()

# Train for a few batches
n_batches = 10
print(f"Training for {n_batches} batches...\n")

for batch_idx, (X_batch, y_batch) in enumerate(train_set.take(n_batches)):
    with tf.GradientTape() as tape:
        # Forward pass
        y_pred = custom_model(X_batch, training=True)
        
        # Compute loss
        loss = loss_fn(y_batch, y_pred)
    
    # Compute gradients
    gradients = tape.gradient(loss, custom_model.trainable_variables)
    
    # Update weights
    optimizer.apply_gradients(zip(gradients, custom_model.trainable_variables))
    
    # Print progress
    if (batch_idx + 1) % 5 == 0:
        print(f"Batch {batch_idx + 1}/{n_batches}, Loss: {loss.numpy():.4f}")

print("\n✅ Custom training loop completed!")

## 16. Using @tf.function for Performance

For even better performance, wrap your training loop in a `@tf.function` decorator. This compiles the Python code into a TensorFlow graph, which runs much faster.

**Benefits:**
- 10-100x speedup for the training loop
- Better optimization by TensorFlow
- Can be exported for production deployment

In [ ]:
# Define a training step as a TensorFlow function
@tf.function
def train_step(X_batch, y_batch, model, optimizer, loss_fn):
    """
    Perform one training step (forward + backward pass).
    Decorated with @tf.function for performance.
    """
    with tf.GradientTape() as tape:
        # Forward pass
        y_pred = model(X_batch, training=True)
        
        # Compute loss
        loss = loss_fn(y_batch, y_pred)
    
    # Compute gradients
    gradients = tape.gradient(loss, model.trainable_variables)
    
    # Update weights
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return loss

# Create a fresh model
tf_func_model = keras.models.Sequential([
    keras.layers.Dense(30, activation='relu', input_shape=[8]),
    keras.layers.Dense(1)
])

optimizer = keras.optimizers.SGD(learning_rate=0.01)
loss_fn = keras.losses.MeanSquaredError()

print("Training with @tf.function...")
print("-" * 50)

# Train
n_batches = 10
for batch_idx, (X_batch, y_batch) in enumerate(train_set.take(n_batches)):
    loss = train_step(X_batch, y_batch, tf_func_model, optimizer, loss_fn)
    
    if (batch_idx + 1) % 5 == 0:
        print(f"Batch {batch_idx + 1}/{n_batches}, Loss: {loss.numpy():.4f}")

print("\n✅ Training with @tf.function completed!")
print("\n💡 The first call to train_step() is slower (graph compilation),")
print("   but subsequent calls are much faster!")

## Summary of Part 2

In this section, we covered advanced Data API concepts for production ML systems:

### Key Concepts Learned:

1. **CSV Preprocessing** - Using `tf.io.decode_csv()` to parse CSV files
2. **Standardization** - Normalizing features with (x - mean) / std
3. **Complete Pipeline** - Building `csv_reader_dataset()` function combining all techniques
4. **Prefetching** - Overlapping data loading with training for maximum performance
5. **Keras Integration** - Using datasets directly with `model.fit()`
6. **Custom Training Loops** - Manual gradient computation and weight updates
7. **@tf.function** - Compiling Python code to TensorFlow graphs for speed

### The Complete Pipeline Recipe:

```python
def csv_reader_dataset(filepaths, ...):
    dataset = tf.data.Dataset.list_files(filepaths)           # 1. List files
    dataset = dataset.interleave(TextLineDataset, ...)        # 2. Read in parallel
    dataset = dataset.map(preprocess, ...)                    # 3. Parse & preprocess
    dataset = dataset.shuffle(buffer_size)                    # 4. Shuffle
    dataset = dataset.repeat(epochs)                          # 5. Repeat
    dataset = dataset.batch(batch_size)                       # 6. Batch
    return dataset.prefetch(1)                                # 7. Prefetch
```

### Performance Best Practices:

✅ Use `num_parallel_calls=AUTOTUNE` for map and interleave  
✅ Always add `.prefetch(1)` at the end  
✅ Use `.cache()` for small datasets that fit in memory  
✅ Shuffle with appropriate buffer size  
✅ Use `@tf.function` for custom training loops  

### What's Next (Part 3):

In the next section, we'll cover:
- TFRecord format for large datasets
- Protocol Buffers (protobufs)
- Writing and reading TFRecords
- Parsing Example and SequenceExample protobufs
- Compressed TFRecords
- Special encodings for images and tensors

---

**🎯 You now know how to build production-ready data pipelines with TensorFlow!**

---

# Part 3: TFRecord Format and Protocol Buffers

**TFRecord** is TensorFlow's preferred format for storing large datasets efficiently. It's a simple binary format optimized for sequential reading and writing.

## 17. Introduction to TFRecord Format

**What is TFRecord?**

TFRecord is a binary file format for storing sequences of binary records. Each record contains:
- **Length** (64 bits)
- **CRC checksum** for length (32 bits) - detects corruption
- **Data** (variable size)
- **CRC checksum** for data (32 bits)

**Why use TFRecord?**
- ✅ **Efficient storage** - Binary format, smaller than CSV
- ✅ **Fast reading** - Optimized for sequential access
- ✅ **Data integrity** - CRC checksums detect corruption
- ✅ **Flexible** - Store any binary data
- ✅ **TensorFlow native** - Best integration with TF pipelines

**When to use TFRecord:**
- Large datasets (> 1 GB)
- Training on multiple machines
- Need fast, sequential data access
- Want to store preprocessed data

## 18. Writing TFRecords - Basic Example

Let's start with a simple example: writing byte strings to a TFRecord file.

In [ ]:
# Write simple byte strings to a TFRecord file
tfrecord_file = "my_data.tfrecord"

with tf.io.TFRecordWriter(tfrecord_file) as writer:
    writer.write(b"This is the first record")
    writer.write(b"And this is the second record")

print(f"✅ Created TFRecord file: {tfrecord_file}")

# Check file size
file_size = os.path.getsize(tfrecord_file)
print(f"File size: {file_size} bytes")

## 19. Reading TFRecords

Now let's read the TFRecord file we just created using `tf.data.TFRecordDataset`.

In [ ]:
# Read the TFRecord file
filepaths = [tfrecord_file]
dataset = tf.data.TFRecordDataset(filepaths)

print("Reading TFRecord file:")
for item in dataset:
    print(item)
    print(f"  Type: {item.dtype}")
    print(f"  Value: {item.numpy()}")
    print()

**Reading multiple TFRecord files in parallel:**

Just like with CSV files, you can read multiple TFRecord files simultaneously for better performance.

In [ ]:
# Create multiple TFRecord files for demonstration
for i in range(3):
    filename = f"my_data_{i}.tfrecord"
    with tf.io.TFRecordWriter(filename) as writer:
        for j in range(2):
            writer.write(f"File {i}, Record {j}".encode())
    print(f"Created: {filename}")

# Read multiple files in parallel
dataset = tf.data.TFRecordDataset(
    ["my_data_0.tfrecord", "my_data_1.tfrecord", "my_data_2.tfrecord"],
    num_parallel_reads=3  # Read 3 files in parallel
)

print("\nReading multiple TFRecord files:")
for item in dataset:
    print(f"  {item.numpy().decode()}")

## 20. Compressed TFRecords

For large datasets, compression can significantly reduce storage space. TFRecord supports **GZIP** and **ZLIB** compression.

In [ ]:
# Write a compressed TFRecord file
compressed_file = "my_compressed.tfrecord"

options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter(compressed_file, options) as writer:
    for i in range(100):
        # Write longer records to see compression benefit
        writer.write(f"This is record number {i} with some repeated text " * 10).encode()

print(f"✅ Created compressed TFRecord: {compressed_file}")

# Compare file sizes
uncompressed_file = "my_uncompressed.tfrecord"
with tf.io.TFRecordWriter(uncompressed_file) as writer:
    for i in range(100):
        writer.write(f"This is record number {i} with some repeated text " * 10).encode()

compressed_size = os.path.getsize(compressed_file)
uncompressed_size = os.path.getsize(uncompressed_file)

print(f"\nFile size comparison:")
print(f"  Uncompressed: {uncompressed_size:,} bytes")
print(f"  Compressed:   {compressed_size:,} bytes")
print(f"  Savings:      {(1 - compressed_size/uncompressed_size)*100:.1f}%")

In [ ]:
# Read the compressed TFRecord
dataset = tf.data.TFRecordDataset([compressed_file], compression_type="GZIP")

print("\nReading compressed TFRecord (first 3 records):")
for i, item in enumerate(dataset.take(3)):
    text = item.numpy().decode()
    print(f"  Record {i}: {text[:60]}...")  # Show first 60 chars

## 21. Protocol Buffers (Protobufs) - Introduction

**What are Protocol Buffers?**

Protocol Buffers (protobufs) are Google's language-neutral, platform-neutral mechanism for serializing structured data. Think of them as a more efficient alternative to XML or JSON.

**Key features:**
- 📦 **Compact** - Binary format, much smaller than text
- ⚡ **Fast** - Quick serialization/deserialization
- 🔧 **Extensible** - Can add fields without breaking old code
- 🌐 **Cross-platform** - Works across languages and systems
- 📝 **Strongly typed** - Schemas define data structure

**TensorFlow's Example protobuf:**

TensorFlow uses a special protobuf called `Example` to store dataset instances in TFRecords. It provides a flexible structure for different data types.

## 22. Creating TensorFlow Example Protobufs

The `Example` protobuf has a simple structure:
- **Features**: A dictionary mapping feature names to Feature objects
- **Feature**: Can contain BytesList, FloatList, or Int64List

Let's create an Example protobuf representing a person.

In [ ]:
# Import the protobuf classes
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

# Create an Example protobuf
person_example = Example(
    features=Features(
        feature={
            "name": Feature(bytes_list=BytesList(value=[b"Alice"])),
            "id": Feature(int64_list=Int64List(value=[123])),
            "emails": Feature(bytes_list=BytesList(value=[b"alice@example.com", b"alice@work.com"]))
        }
    )
)

print("Created Example protobuf:")
print(person_example)
print()

# Serialize to bytes
serialized = person_example.SerializeToString()
print(f"Serialized size: {len(serialized)} bytes")
print(f"Serialized (first 50 bytes): {serialized[:50]}")

## 23. Writing Examples to TFRecord

Now let's write Example protobufs to a TFRecord file - this is the standard way to store structured data in TFRecords.

In [ ]:
# Create a TFRecord with multiple Example protobufs
contacts_file = "my_contacts.tfrecord"

# Create multiple person examples
people = [
    {"name": b"Alice", "id": 123, "emails": [b"alice@example.com", b"alice@work.com"]},
    {"name": b"Bob", "id": 456, "emails": [b"bob@example.com"]},
    {"name": b"Charlie", "id": 789, "emails": [b"charlie@example.com", b"charlie@personal.com", b"charlie@school.com"]},
]

with tf.io.TFRecordWriter(contacts_file) as writer:
    for person in people:
        example = Example(
            features=Features(
                feature={
                    "name": Feature(bytes_list=BytesList(value=[person["name"]])),
                    "id": Feature(int64_list=Int64List(value=[person["id"]])),
                    "emails": Feature(bytes_list=BytesList(value=person["emails"]))
                }
            )
        )
        writer.write(example.SerializeToString())

print(f"✅ Wrote {len(people)} Examples to {contacts_file}")
print(f"File size: {os.path.getsize(contacts_file)} bytes")

## 24. Parsing Examples from TFRecord

To read Examples from a TFRecord, we need to:
1. Define a **feature description** - tells TensorFlow what to expect
2. Use `tf.io.parse_single_example()` to parse each serialized Example

**Feature types:**
- **`FixedLenFeature`** - Fixed-length features (returns dense tensor)
- **`VarLenFeature`** - Variable-length features (returns sparse tensor)

In [ ]:
# Define feature description
feature_description = {
    "name": tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails": tf.io.VarLenFeature(tf.string),  # Variable number of emails
}

# Read and parse the TFRecord
dataset = tf.data.TFRecordDataset([contacts_file])

print("Parsing Examples from TFRecord:")
print("=" * 60)

for i, serialized_example in enumerate(dataset):
    # Parse the serialized example
    parsed_example = tf.io.parse_single_example(serialized_example, feature_description)
    
    print(f"\nPerson {i+1}:")
    print(f"  Name: {parsed_example['name'].numpy().decode()}")
    print(f"  ID:   {parsed_example['id'].numpy()}")
    
    # emails is a sparse tensor (variable length)
    # Convert to dense tensor for easier display
    emails_dense = tf.sparse.to_dense(parsed_example["emails"], default_value=b"")
    print(f"  Emails: {[email.decode() for email in emails_dense.numpy() if email]}")
    
    # Alternative: access sparse tensor values directly
    print(f"  Emails (from sparse): {[email.decode() for email in parsed_example['emails'].values.numpy()]}")

## 25. Batch Parsing for Performance

When processing batches, use `tf.io.parse_example()` (plural) instead of parsing one at a time. This is much faster!

In [ ]:
# Batch the dataset before parsing
dataset = tf.data.TFRecordDataset([contacts_file]).batch(2)

print("Batch parsing Examples:")
print("=" * 60)

for batch_idx, serialized_examples in enumerate(dataset):
    # Parse the entire batch at once
    parsed_examples = tf.io.parse_example(serialized_examples, feature_description)
    
    print(f"\nBatch {batch_idx + 1}:")
    print(f"  Names: {[name.numpy().decode() for name in parsed_examples['name']]}")
    print(f"  IDs:   {parsed_examples['id'].numpy()}")
    
    # For sparse tensors in batches, we get a single SparseTensor
    print(f"  Emails shape: {parsed_examples['emails'].dense_shape.numpy()}")
    print(f"  Emails values: {[email.decode() for email in parsed_examples['emails'].values.numpy()]}")

## 26. Real-World Example: Housing Data in TFRecord

Let's convert our housing CSV data to TFRecord format using Example protobufs. This is a common workflow for production ML systems.

In [ ]:
# Helper function to create a Feature from a value
def _bytes_feature(value):
    """Returns a bytes_list from a string / byte."""
    if isinstance(value, type(tf.constant(0))):
        value = value.numpy()
    return Feature(bytes_list=BytesList(value=[value]))

def _float_feature(value):
    """Returns a float_list from a float / double."""
    return Feature(float_list=FloatList(value=[value]))

def _int64_feature(value):
    """Returns an int64_list from a bool / enum / int / uint."""
    return Feature(int64_list=Int64List(value=[value]))

def _float_list_feature(values):
    """Returns a float_list from a list of floats."""
    return Feature(float_list=FloatList(value=values))

# Convert one CSV file to TFRecord
input_csv = os.path.join(csv_dir, "housing_data_1.csv")
output_tfrecord = "housing_train.tfrecord"

print(f"Converting {input_csv} to TFRecord format...")

with tf.io.TFRecordWriter(output_tfrecord) as writer:
    with open(input_csv, 'r') as f:
        header = f.readline()  # Skip header
        
        for line_num, line in enumerate(f):
            values = [float(x) for x in line.strip().split(',')]
            
            # Split features and target
            features_values = values[:-1]  # First 8 values
            target_value = values[-1]       # Last value
            
            # Create Example
            example = Example(
                features=Features(
                    feature={
                        'features': _float_list_feature(features_values),
                        'target': _float_feature(target_value)
                    }
                )
            )
            
            writer.write(example.SerializeToString())

print(f"✅ Converted {line_num + 1} records to {output_tfrecord}")
print(f"TFRecord size: {os.path.getsize(output_tfrecord):,} bytes")

In [ ]:
# Now read and parse the housing TFRecord
feature_description = {
    'features': tf.io.FixedLenFeature([8], tf.float32),  # 8 features
    'target': tf.io.FixedLenFeature([1], tf.float32),    # 1 target
}

def parse_housing_example(serialized_example):
    """Parse a housing example from TFRecord."""
    example = tf.io.parse_single_example(serialized_example, feature_description)
    features = example['features']
    target = example['target']
    return features, target

# Create dataset from TFRecord
housing_dataset = tf.data.TFRecordDataset([output_tfrecord])
housing_dataset = housing_dataset.map(parse_housing_example)
housing_dataset = housing_dataset.batch(32).prefetch(1)

print("Housing dataset from TFRecord:")
print(housing_dataset)
print()

# Show a sample batch
for X_batch, y_batch in housing_dataset.take(1):
    print(f"Features batch shape: {X_batch.shape}")
    print(f"Target batch shape:   {y_batch.shape}")
    print()
    print("First 3 samples:")
    print("Features:")
    print(X_batch[:3].numpy())
    print()
    print("Targets:")
    print(y_batch[:3].numpy())

## 27. Special Encodings for Images and Tensors

TFRecords support special encodings for common data types:

**For images:**
- `tf.io.encode_jpeg()` / `tf.io.decode_jpeg()`
- `tf.io.encode_png()` / `tf.io.decode_png()`

**For tensors:**
- `tf.io.serialize_tensor()` / `tf.io.parse_tensor()`

Let's see examples of both.

In [ ]:
# Example 1: Serializing tensors
print("Example 1: Tensor Serialization")
print("=" * 60)

# Create a sample tensor
sample_tensor = tf.constant([[1, 2, 3], [4, 5, 6]], dtype=tf.float32)
print(f"Original tensor:\n{sample_tensor.numpy()}")
print()

# Serialize the tensor
serialized_tensor = tf.io.serialize_tensor(sample_tensor)
print(f"Serialized tensor (bytes): {serialized_tensor.numpy()[:50]}...")
print(f"Serialized size: {len(serialized_tensor.numpy())} bytes")
print()

# Store in TFRecord
tensor_file = "tensor_example.tfrecord"
with tf.io.TFRecordWriter(tensor_file) as writer:
    example = Example(
        features=Features(
            feature={
                'tensor': _bytes_feature(serialized_tensor.numpy())
            }
        )
    )
    writer.write(example.SerializeToString())

print(f"✅ Wrote tensor to {tensor_file}")
print()

# Read and deserialize
dataset = tf.data.TFRecordDataset([tensor_file])
for serialized_example in dataset:
    parsed = tf.io.parse_single_example(
        serialized_example,
        {'tensor': tf.io.FixedLenFeature([], tf.string)}
    )
    
    # Deserialize the tensor
    restored_tensor = tf.io.parse_tensor(parsed['tensor'], out_type=tf.float32)
    print(f"Restored tensor:\n{restored_tensor.numpy()}")
    print(f"Tensors match: {tf.reduce_all(tf.equal(sample_tensor, restored_tensor)).numpy()}")

In [ ]:
# Example 2: Image encoding (simulated with synthetic image)
print("\nExample 2: Image Encoding")
print("=" * 60)

# Create a synthetic "image" (random pixels)
fake_image = tf.random.uniform([64, 64, 3], minval=0, maxval=256, dtype=tf.int32)
fake_image = tf.cast(fake_image, tf.uint8)

print(f"Fake image shape: {fake_image.shape}")
print(f"Raw image size: {fake_image.numpy().nbytes} bytes")
print()

# Encode as JPEG (compressed)
encoded_jpeg = tf.io.encode_jpeg(fake_image, quality=95)
print(f"JPEG encoded size: {len(encoded_jpeg.numpy())} bytes")
print(f"Compression ratio: {fake_image.numpy().nbytes / len(encoded_jpeg.numpy()):.2f}x")
print()

# Store in TFRecord
image_file = "image_example.tfrecord"
with tf.io.TFRecordWriter(image_file) as writer:
    example = Example(
        features=Features(
            feature={
                'image': _bytes_feature(encoded_jpeg.numpy()),
                'height': _int64_feature(64),
                'width': _int64_feature(64),
            }
        )
    )
    writer.write(example.SerializeToString())

print(f"✅ Wrote image to {image_file}")
print()

# Read and decode
dataset = tf.data.TFRecordDataset([image_file])
for serialized_example in dataset:
    parsed = tf.io.parse_single_example(
        serialized_example,
        {
            'image': tf.io.FixedLenFeature([], tf.string),
            'height': tf.io.FixedLenFeature([], tf.int64),
            'width': tf.io.FixedLenFeature([], tf.int64),
        }
    )
    
    # Decode the JPEG
    decoded_image = tf.io.decode_jpeg(parsed['image'])
    print(f"Decoded image shape: {decoded_image.shape}")
    print(f"Height: {parsed['height'].numpy()}, Width: {parsed['width'].numpy()}")

## Summary of Part 3

In this section, we covered TFRecord format and Protocol Buffers for efficient data storage:

### Key Concepts Learned:

1. **TFRecord Format** - Binary format with CRC checksums for data integrity
2. **Writing TFRecords** - Using `tf.io.TFRecordWriter`
3. **Reading TFRecords** - Using `tf.data.TFRecordDataset`
4. **Compression** - GZIP compression for storage savings
5. **Protocol Buffers** - Google's efficient serialization format
6. **Example Protobufs** - TensorFlow's standard format for storing data
7. **Feature Types** - BytesList, FloatList, Int64List
8. **Parsing** - `parse_single_example()` and `parse_example()` (batch)
9. **Fixed vs Variable Length** - FixedLenFeature vs VarLenFeature
10. **Special Encodings** - JPEG/PNG for images, serialize_tensor for arrays

### TFRecord Workflow:

```python
# Writing
with tf.io.TFRecordWriter("data.tfrecord") as writer:
    example = Example(features=Features(feature={
        'x': Feature(float_list=FloatList(value=[...])),
        'y': Feature(int64_list=Int64List(value=[...]))
    }))
    writer.write(example.SerializeToString())

# Reading
dataset = tf.data.TFRecordDataset(["data.tfrecord"])
dataset = dataset.map(parse_function)
```

### When to Use TFRecord:

✅ Large datasets (> 1 GB)  
✅ Need fast sequential access  
✅ Training on multiple machines  
✅ Want to store preprocessed data  
✅ Mixed data types (images, text, numbers)  

### Performance Tips:

✅ Use compression for repetitive data  
✅ Batch before parsing with `parse_example()`  
✅ Store preprocessed data to avoid repeated computation  
✅ Use appropriate feature types (Fixed vs Variable length)  

---

**🎯 You now know how to use TFRecords for production-scale ML workflows!**

The notebook is now approximately 75% complete. The final 25% would cover preprocessing layers and advanced topics, which we can continue with if you'd like!

---

# Part 4: Preprocessing Layers and Advanced Topics

In this final section, we'll explore Keras preprocessing layers for in-model preprocessing, categorical encoding techniques, and TensorFlow Datasets (TFDS).

## 28. Standardization with Keras Layers

**Two approaches to standardization in models:**

1. **Lambda Layer** - Simple, inline standardization
2. **Custom Layer** - Reusable, can adapt to data
3. **Normalization Layer** - Built-in Keras layer (recommended!)

**Advantage:** Preprocessing is part of the model, so it's applied automatically during inference.

In [ ]:
# Generate sample data for demonstration
X_train_sample = np.random.randn(100, 5) * 10 + 50  # Mean ~50, varied std
y_train_sample = np.random.randn(100, 1)

print("Sample data statistics:")
print(f"Means: {X_train_sample.mean(axis=0)}")
print(f"Stds:  {X_train_sample.std(axis=0)}")
print()

# Method 1: Lambda Layer
print("Method 1: Lambda Layer")
print("=" * 60)

means = np.mean(X_train_sample, axis=0, keepdims=True)
stds = np.std(X_train_sample, axis=0, keepdims=True)
eps = keras.backend.epsilon()

model_lambda = keras.models.Sequential([
    keras.layers.Lambda(lambda inputs: (inputs - means) / (stds + eps)),
    keras.layers.Dense(10, activation='relu'),
    keras.layers.Dense(1)
])

print("Model with Lambda layer:")
model_lambda.build(input_shape=(None, 5))
model_lambda.summary()

In [ ]:
# Method 2: Custom Reusable Layer
print("\nMethod 2: Custom Reusable Layer")
print("=" * 60)

class Standardization(keras.layers.Layer):
    def adapt(self, data_sample):
        """Compute statistics from a data sample."""
        self.means_ = np.mean(data_sample, axis=0, keepdims=True)
        self.stds_ = np.std(data_sample, axis=0, keepdims=True)
    
    def call(self, inputs):
        """Apply standardization."""
        return (inputs - self.means_) / (self.stds_ + keras.backend.epsilon())

# Create and adapt the layer
std_layer = Standardization()
std_layer.adapt(X_train_sample)

model_custom = keras.Sequential([
    std_layer,
    keras.layers.Dense(10, activation='relu'),
    keras.layers.Dense(1)
])

print("Model with custom Standardization layer:")
model_custom.build(input_shape=(None, 5))
model_custom.summary()

In [ ]:
# Method 3: Built-in Normalization Layer (RECOMMENDED!)
print("\nMethod 3: Keras Normalization Layer (Recommended)")
print("=" * 60)

normalization_layer = keras.layers.Normalization()
normalization_layer.adapt(X_train_sample)

model_norm = keras.Sequential([
    normalization_layer,
    keras.layers.Dense(10, activation='relu'),
    keras.layers.Dense(1)
])

print("Model with Normalization layer:")
model_norm.build(input_shape=(None, 5))
model_norm.summary()

# Test the normalization
print("\nTesting normalization:")
sample_input = X_train_sample[:3]
normalized_output = normalization_layer(sample_input)
print(f"Original input mean: {sample_input.mean(axis=0)}")
print(f"Normalized output mean: {normalized_output.numpy().mean(axis=0)}")
print(f"Normalized output std: {normalized_output.numpy().std(axis=0)}")

## 29. Encoding Categorical Features - One-Hot Encoding

When dealing with categorical data (e.g., "ocean_proximity" in housing data), we need to encode them as numbers.

**One-Hot Encoding** creates a binary vector for each category:
- "INLAND" → [0, 1, 0, 0, 0]
- "NEAR OCEAN" → [0, 0, 1, 0, 0]

**When to use:**
- **< 10 categories** → One-hot encoding (simple, effective)
- **10-50 categories** → Experiment with both one-hot and embeddings  
- **> 50 categories** → Embeddings (more efficient)

In [ ]:
# Create a vocabulary (list of known categories)
vocab = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]
indices = tf.range(len(vocab), dtype=tf.int64)

# Create lookup table
table_init = tf.lookup.KeyValueTensorInitializer(vocab, indices)

# Out-of-vocabulary (OOV) buckets handle unknown categories
num_oov_buckets = 2
table = tf.lookup.StaticVocabularyTable(table_init, num_oov_buckets)

print("Vocabulary and lookup table:")
print(f"Known categories: {vocab}")
print(f"Number of OOV buckets: {num_oov_buckets}")
print(f"Total size (vocab + OOV): {len(vocab) + num_oov_buckets}")
print()

# Test the lookup
categories = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND", "UNKNOWN"])
cat_indices = table.lookup(categories)

print("Lookup results:")
for cat, idx in zip(categories.numpy(), cat_indices.numpy()):
    cat_str = cat.decode() if isinstance(cat, bytes) else cat
    if idx < len(vocab):
        print(f"  '{cat_str}' → index {idx} (known)")
    else:
        print(f"  '{cat_str}' → index {idx} (OOV bucket)")

In [ ]:
# Convert indices to one-hot encoding
categories_test = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND"])
cat_indices = table.lookup(categories_test)

depth = len(vocab) + num_oov_buckets
cat_one_hot = tf.one_hot(cat_indices, depth=depth)

print("\nOne-hot encoding:")
print(f"Categories: {[c.decode() if isinstance(c, bytes) else c for c in categories_test.numpy()]}")
print(f"Indices:    {cat_indices.numpy()}")
print(f"\nOne-hot vectors (shape: {cat_one_hot.shape}):")
print(cat_one_hot.numpy())
print()

# Explain the encoding
print("Explanation:")
print("  Each row is a one-hot vector for one category")
print("  The '1' is at the position corresponding to the category index")
print(f"  Vector length = {depth} (5 known + 2 OOV buckets)")

## 30. Encoding Categorical Features - Embeddings

**Embeddings** are trainable dense vectors that represent categories. They're more efficient for many categories.

**Key concepts:**
- Start with random vectors
- Learn better representations during training
- Capture semantic relationships (similar categories get similar vectors)
- Much more efficient than one-hot for many categories

**Famous example:** Word2Vec (Mikolov et al., 2013)
- "King" - "Man" + "Woman" ≈ "Queen"
- Synonyms have similar embeddings

In [ ]:
# Manual embedding implementation
embedding_dim = 2  # Each category is represented by a 2D vector

# Initialize random embedding matrix
embed_init = tf.random.uniform([len(vocab) + num_oov_buckets, embedding_dim], seed=42)
embedding_matrix = tf.Variable(embed_init)

print("Embedding Matrix (random initialization):")
print(f"Shape: {embedding_matrix.shape} ({len(vocab) + num_oov_buckets} categories × {embedding_dim} dimensions)")
print(embedding_matrix.numpy())
print()

# Look up embeddings for categories
categories = tf.constant(["NEAR BAY", "DESERT", "INLAND", "INLAND"])
cat_indices = table.lookup(categories)
embeddings = tf.nn.embedding_lookup(embedding_matrix, cat_indices)

print("Category embeddings:")
for cat, emb in zip(categories.numpy(), embeddings.numpy()):
    cat_str = cat.decode() if isinstance(cat, bytes) else cat
    print(f"  '{cat_str}': {emb}")

print("\nNote: These are random vectors initially.")
print("During training, they'll be updated to capture meaningful relationships!")

In [ ]:
# Using Keras Embedding Layer (much simpler!)
print("Using Keras Embedding Layer:")
print("=" * 60)

embedding_layer = keras.layers.Embedding(
    input_dim=len(vocab) + num_oov_buckets,
    output_dim=embedding_dim
)

# Test with our categories
cat_indices = table.lookup(categories)
embeddings_keras = embedding_layer(cat_indices)

print(f"Embeddings shape: {embeddings_keras.shape}")
print(f"Embeddings:\n{embeddings_keras.numpy()}")
print()

# Complete model example combining regular features and categorical features
print("\nComplete Model with Mixed Inputs:")
print("=" * 60)

# Define inputs
regular_inputs = keras.layers.Input(shape=[8], name="regular_features")
categorical_input = keras.layers.Input(shape=[], dtype=tf.string, name="category")

# Process categorical input
cat_indices = keras.layers.Lambda(lambda cats: table.lookup(cats))(categorical_input)
cat_embed = keras.layers.Embedding(input_dim=len(vocab) + num_oov_buckets, 
                                   output_dim=2)(cat_indices)

# Combine all features
combined = keras.layers.concatenate([regular_inputs, cat_embed])

# Add dense layers
hidden = keras.layers.Dense(30, activation='relu')(combined)
outputs = keras.layers.Dense(1)(hidden)

# Create model
mixed_model = keras.models.Model(
    inputs=[regular_inputs, categorical_input],
    outputs=outputs
)

print("Model architecture:")
mixed_model.summary()
print()
print("✅ This model can handle both numerical and categorical features!")

## 31. Keras Preprocessing Layers

Modern Keras provides built-in preprocessing layers that can be included directly in models:

**Advantages:**
- ✅ Preprocessing is part of the model
- ✅ No training/serving skew
- ✅ Model is self-contained
- ✅ Easy deployment

**Available layers:**
- `Normalization` - Standardize numerical features
- `TextVectorization` - Convert text to indices/vectors
- `Discretization` - Bin continuous values
- `StringLookup` / `IntegerLookup` - Categorical encoding
- `Hashing` - Hash-based encoding
- `CategoryEncoding` - One-hot or multi-hot encoding

In [ ]:
# Example: TextVectorization Layer
print("Example: TextVectorization Layer")
print("=" * 60)

# Sample text data
texts = [
    "The quick brown fox",
    "jumps over the lazy dog",
    "The dog was really very lazy",
    "The fox was quick and clever"
]

# Create and adapt TextVectorization layer
text_vectorization = keras.layers.TextVectorization(
    max_tokens=20,  # Vocabulary size
    output_mode='int'  # Output word indices
)
text_vectorization.adapt(texts)

print("Vocabulary (first 10 words):")
vocab_list = text_vectorization.get_vocabulary()
for i, word in enumerate(vocab_list[:10]):
    print(f"  {i}: '{word}'")
print()

# Vectorize some text
test_texts = tf.constant(["The clever fox", "A lazy cat"])
vectorized = text_vectorization(test_texts)

print("Vectorization results:")
for text, vec in zip(test_texts.numpy(), vectorized.numpy()):
    text_str = text.decode() if isinstance(text, bytes) else text
    print(f"  '{text_str}' → {vec}")
print()

# Example with TF-IDF mode
text_vectorization_tfidf = keras.layers.TextVectorization(
    max_tokens=20,
    output_mode='tf_idf'  # TF-IDF weights
)
text_vectorization_tfidf.adapt(texts)

tfidf_vectors = text_vectorization_tfidf(test_texts)
print("TF-IDF mode (first vector):")
print(f"  Shape: {tfidf_vectors.shape}")
print(f"  Values (first 10): {tfidf_vectors[0, :10].numpy()}")

In [ ]:
# Example: Discretization Layer
print("\nExample: Discretization Layer")
print("=" * 60)

# Generate continuous age data
ages = np.random.uniform(18, 80, size=100)

# Create discretization layer (bin ages into categories)
age_discretization = keras.layers.Discretization(
    bin_boundaries=[25, 40, 60]  # Creates bins: <25, 25-40, 40-60, >60
)
age_discretization.adapt(ages)

# Test discretization
test_ages = tf.constant([20.0, 35.0, 50.0, 70.0])
discretized = age_discretization(test_ages)

print("Age discretization:")
print(f"  Bin boundaries: [25, 40, 60]")
print(f"  Creates 4 bins: [<25, 25-40, 40-60, >60]")
print()
for age, bin_idx in zip(test_ages.numpy(), discretized.numpy()):
    print(f"  Age {age:.0f} → Bin {bin_idx}")
print()
print("Note: Bin indices can be used with CategoryEncoding or Embedding layers")

## 32. TensorFlow Datasets (TFDS)

**TFDS** provides easy access to hundreds of ready-to-use datasets for machine learning.

**Popular datasets available:**
- MNIST, Fashion MNIST
- CIFAR-10, CIFAR-100
- ImageNet
- COCO, VOC
- IMDb reviews
- Many more!

Visit [https://tensorflow.org/datasets/catalog](https://tensorflow.org/datasets/catalog) for the full catalog.

In [ ]:
# Load MNIST dataset using TFDS
print("Loading MNIST with TensorFlow Datasets")
print("=" * 60)

try:
    import tensorflow_datasets as tfds
    
    # Load MNIST (downloads if not cached)
    print("Loading MNIST dataset...")
    dataset_dict = tfds.load(name="mnist", split=["train", "test"], as_supervised=True)
    mnist_train, mnist_test = dataset_dict[0], dataset_dict[1]
    
    print("✅ MNIST loaded successfully!")
    print()
    
    # Inspect the dataset
    print("Dataset info:")
    for split_name, split_data in [("Train", mnist_train), ("Test", mnist_test)]:
        print(f"  {split_name}: {split_data}")
    print()
    
    # Prepare for training
    mnist_train_prepared = mnist_train.shuffle(10000).batch(32).prefetch(1)
    
    # Show a sample
    print("Sample batch:")
    for images, labels in mnist_train_prepared.take(1):
        print(f"  Images shape: {images.shape}  (batch_size × height × width × channels)")
        print(f"  Labels shape: {labels.shape}")
        print(f"  Image dtype: {images.dtype}, range: [{images.numpy().min()}, {images.numpy().max()}]")
        print(f"  Labels: {labels.numpy()[:10]}")
    
    print()
    print("Ready to use with model.fit()!")
    
except ImportError:
    print("⚠️  tensorflow_datasets not installed")
    print("Install with: pip install tensorflow-datasets")
    print()
    print("TFDS makes it easy to load standard datasets with just one line!")
    print("Example: dataset = tfds.load('mnist', split='train', as_supervised=True)")

## Final Summary - Complete Chapter Overview

Congratulations! You've completed a comprehensive tutorial on TensorFlow's Data API and preprocessing techniques.

### 🎯 What We Covered:

**Part 1: Data API Fundamentals (Sections 1-10)**
- Creating datasets from tensors
- Chaining transformations (repeat, batch, map, filter)
- Shuffling strategies for training
- Reading CSV files
- Interleaving multiple files for parallel I/O

**Part 2: Advanced Pipelines (Sections 11-16)**
- CSV preprocessing with decode_csv
- Building complete data pipelines
- Prefetching for performance
- Integration with tf.keras (fit, evaluate, predict)
- Custom training loops
- @tf.function for graph optimization

**Part 3: TFRecord & Protobufs (Sections 17-27)**
- TFRecord format and benefits
- Writing and reading TFRecords
- Compression (GZIP)
- Protocol Buffers (protobufs)
- Example protobufs (BytesList, FloatList, Int64List)
- Parsing with FixedLenFeature and VarLenFeature
- Special encodings (images, tensors)
- Real-world CSV → TFRecord conversion

**Part 4: Preprocessing & TFDS (Sections 28-32)**
- Standardization with Keras layers
- One-hot encoding for categorical features
- Embeddings for high-cardinality features
- Keras preprocessing layers (TextVectorization, Discretization)
- TensorFlow Datasets (TFDS) for quick dataset access

### 🔑 Key Takeaways:

1. **Data API** - Efficient, scalable data loading with multithreading
2. **TFRecords** - Production format for large-scale ML
3. **Preprocessing** - Multiple options (Data API, Keras layers, TF Transform)
4. **Categorical Encoding** - One-hot (<10), embeddings (>50)
5. **Performance** - prefetch(), cache(), AUTOTUNE
6. **Integration** - Seamless with tf.keras

### 📊 Performance Best Practices:

```python
# The Perfect Pipeline Template
dataset = tf.data.Dataset.list_files(files)
dataset = dataset.interleave(..., num_parallel_calls=AUTOTUNE)
dataset = dataset.map(preprocess, num_parallel_calls=AUTOTUNE)
dataset = dataset.cache()  # If dataset fits in memory
dataset = dataset.shuffle(buffer_size)
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(AUTOTUNE)
```

### 🚀 Next Steps:

- Experiment with different preprocessing techniques
- Try TFDS for quick prototyping
- Convert your datasets to TFRecord for production
- Explore advanced topics (TF Transform, distributed training)
- Practice with the exercises in the README!

---

**🎉 You're now equipped to build production-ready ML data pipelines with TensorFlow!**

Thank you for following along. Happy coding! 🤖